In [ ]:
# 04 Ranking-aware experiment (Experiment B)
#
# Question. Does a pairwise ranking term change ordering quality relative to A?
#
# Type. Pair diagnostics (labels only) plus run/inspection (spec). Config values are untuned placeholders until the declared grid is run on validation data.
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)

import dataclasses
import json
import logging

import matplotlib.pyplot as plt
import torch

from ct_iqa.config.loader import load_study_config
from ct_iqa.experiment import report
from ct_iqa.experiment.run import run_experiment
from ct_iqa.experiment.runs import select_run

CONFIG = "configs/experiments/b_ranking_aware.yaml"
cfg = load_study_config(CONFIG)
print(cfg.run.experiment_id, "|", cfg.model.name, "|", cfg.loss.name, "| bn_eps =", cfg.model.bn_eps,
      "| epochs =", cfg.train.epochs, "| batch =", cfg.train.batch_size, "| seed =", cfg.train.seed)

import numpy as np
import pandas as pd


def show(title, obj):
    """Print a section title, then the table as plain text."""
    print(f"\n=== {title} ===")
    print(obj.to_string())   # plain text table, no interactive viewer



In [ ]:
# Pair diagnostics (training labels only)
# How many in-batch pairs are eligible for each score-difference threshold, for one random batch of 64 training labels.
from ct_iqa.data.sampling import ordered_pair_mask

labels = json.loads(Path("data/labels/ldct_iqac/train.json").read_text())
manifest = json.loads(Path("experiments/study_v2/protocol/split_manifest.json").read_text())
rng = np.random.default_rng(0)
batch = torch.tensor([labels[n] for n in rng.choice(manifest["train"], 64, replace=False)])
total = 64 * 63 // 2
rows = [{"min_score_diff": d, "valid pairs": int(ordered_pair_mask(batch, d).sum())} for d in (0.0, 0.25, 0.5, 1.0)]
pairs = pd.DataFrame(rows).assign(**{"fraction of all pairs": lambda df: (df["valid pairs"] / total).round(3)})
show("Eligible in-batch pairs by minimum score difference (64 training labels)", pairs)
print("configured:", cfg.loss.ranking)


In [ ]:
# Run (explicit, expensive)
# Set RUN_TRAINING = True only with an approved compute budget; this trains for the configured number of epochs and writes a new immutable run directory.
# Training is OFF by default. Set RUN_TRAINING = True only with an approved compute budget: it trains for
# the configured number of epochs and writes a NEW immutable run directory. Progress is logged one line per epoch below the cell.
RUN_TRAINING = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN_ID = None      # name of a completed run to inspect; required once several completed runs exist

if RUN_TRAINING:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)
    cfg = dataclasses.replace(cfg, train=dataclasses.replace(cfg.train, device=DEVICE))
    run_dir = run_experiment(cfg, config_path=CONFIG)
else:
    run_dir = select_run(cfg.run.experiment_id, RUN_ID)
print("device:", DEVICE, "| run directory:", run_dir if run_dir else "NO COMPLETED RUN FOUND (status PENDING)")


In [ ]:
# Results
if run_dir:
    history = json.loads((run_dir / "history.json").read_text())
    metrics = json.loads((run_dir / "metrics_validation.json").read_text())
    show(f"Validation metrics at selected epoch {history['best_epoch']} (policy: {history['selection_policy']}; "
         "raw 0-4 scale; selection-biased)", report.metrics_frame(metrics))
    show("Per-epoch history (selected = True marks the chosen epoch)", report.history_frame(history).round(4))
    report.plot_history(history, cfg.run.experiment_id)
    plt.show()


In [ ]:
if run_dir:
    parts = report.loss_parts_frame(history)
    show("Training loss components per epoch", parts.round(4))
    fig, ax = plt.subplots(figsize=(5.5, 3.4))
    parts[["mse", "rank"]].plot(ax=ax)
    ax.set(xlabel="epoch (1-based)", ylabel="mean per-batch value", title="Training loss components")
    ax.grid(alpha=0.3)
    plt.show()
